In [ ]:
import requests
import json
import time
from datetime import datetime
import os

In [ ]:
API_URL = "https://ckan2.multimediagdansk.pl/gpsPositions?v=2"
#LANDING = "/Volumes/dbr_dev/live_transit_monitor/project_volume/streaming/gps_landing"
LANDING = "gps_landing"

In [ ]:
def fetch_snapshot():
    """Fetching the latest snapshot of GPS data from the API and saving it to a JSON file."""
    print("Fetching API data...")

    try:
        r = requests.get(API_URL, timeout=10)
        r.raise_for_status()
        return r.json()

    except Exception as e:
        print(f"Error occurred: {e}")
        return None

In [ ]:
def save_snapshot(data):
    """Saving the fetched GPS data to a JSON file in the landing directory."""
    try:
        os.makedirs(LANDING, exist_ok=True)
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        file_name = f"gps_{ts}.json"
        file_path = os.path.join(LANDING, file_name)
        with open(file_path, "w", encoding="utf-8") as f:
            for v in data["vehicles"]:
                v["lastUpdate"] = data["lastUpdate"] # Add lastUpdate to each vehicle entry
                f.write(json.dumps(v, ensure_ascii=False) + "\n") # Write each vehicle entry as a separate line in the JSON file
        print(f"Data saved to {file_path}")
        return True

    except Exception as e:
        print(f"Error occurred while saving: {e}")
        return False

In [ ]:
for attempt in range(5):
    data = fetch_snapshot()
    if data and save_snapshot(data):
        break  # Exit the loop if successful
    time.sleep(5)  # Wait for 5 seconds before retrying
else:
    print("All attempts failed; no data was saved.")